# View fMRIPrep Report

Renders a participant's fMRIPrep HTML report inline in the notebook.

fMRIPrep generates one `.html` report per subject in the derivatives directory. This notebook loads that file and displays it here so you can review registration quality, brain masks, carpet plots, and confound traces without leaving Jupyter.

**Before running:** fMRIPrep must have completed successfully for the subject you want to review.

## 1. Imports

In [ ]:
%matplotlib inline
import os
import re
from IPython.display import HTML
from bs4 import BeautifulSoup

## 2. Set Subject

Set `sub_id` to the participant you want to review. All paths are derived from the notebook's location.

In [ ]:
# Subject to review — include the 'sub-' prefix
sub_id = 'sub-GEO053'

# Derive paths from notebook location (scripts/FMRIPREP/)
project_dir    = os.path.abspath('../../')
derivatives_dir = os.path.join(project_dir, 'data/bids_data/derivatives_nocorrection')
report_path    = os.path.join(derivatives_dir, f'{sub_id}.html')

print(f'Looking for report: {report_path}')
print(f'Report exists: {os.path.exists(report_path)}')

## 3. Render Report

The fMRIPrep HTML report links to external SVG files using relative paths. This cell rewrites those links to point to the actual file locations so they render correctly in the notebook.

In [ ]:
if not os.path.exists(report_path):
    raise FileNotFoundError(
        f'Report not found: {report_path}\n'
        f'Check that fMRIPrep completed for {sub_id} and that derivatives_dir is correct.'
    )

html = open(report_path).read()
dom = BeautifulSoup(html, 'html.parser')

# Rewrite relative hrefs to absolute paths so embedded SVGs render correctly
for obj in dom.find_all(href=re.compile(r'^\./')):
    abs_path = re.sub(r'^\.\./', derivatives_dir + '/', obj['href'])
    abs_path = re.sub(r'^\.',  derivatives_dir,       abs_path)
    embed_tag = BeautifulSoup(f'<embed src="{abs_path}"/>', 'html.parser')
    obj.replaceWith(embed_tag)

HTML(str(dom))